## <center> **GDELT 2.0 MULTILANGUAL**   
## <center> **MOROCCO**

## **1. EXTRACTION**

In [1]:
import os
import io
import re
import zipfile
import requests
import pandas as pd
from datetime import datetime
# Google Sheets
import gspread
from gspread_dataframe import set_with_dataframe

In [ ]:
#1 CONFIGURATION
TARGET_YEAR_MONTH = "202607"  # YYYYMM 
MOROCCO_CAMEO_CODE = "MAR"   # CAMEO Morocco
MOROCCO_FIPS_CODE = "MO"      # FIPS Morrocan geography
OUTPUT_MASTER_CSV = f"MAR{TARGET_YEAR_MONTH}_multilingual.csv"
GDELT_HEADERS = ["GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate", "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode", "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code", 
                 "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code", "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode", "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code", 
                 "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code", "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode", "QuadClass",  "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone", 
                 "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode", "Actor1Geo_ADM1Code", "Actor1Geo_ADM2Code", "Actor1Geo_Lat", "Actor1Geo_Long", "Actor1Geo_FeatureID","Actor2Geo_Type", "Actor2Geo_FullName", 
                 "Actor2Geo_CountryCode", "Actor2Geo_ADM1Code", "Actor2Geo_ADM2Code", "Actor2Geo_Lat", "Actor2Geo_Long", "Actor2Geo_FeatureID", "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode", 
                 "ActionGeo_ADM1Code", "ActionGeo_ADM2Code",  "ActionGeo_Lat", "ActionGeo_Long", "ActionGeo_FeatureID", "DATEADDED", "SOURCEURL"]

def get_file_list():
    """Fetches the complete GDELT 2.0 file distribution list."""
    print("Fetching master GDELT 2.0 file list...")
    url = "http://data.gdeltproject.org/gdeltv2/masterfilelist-translation.txt"
    response = requests.get(url)
    response.raise_for_status()
    return response.text.splitlines()

def filter_events(master_list):
    """Filters the file list to only grab export (event) files from Target_Year_Month."""
    target_urls = []
    for line in master_list:
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) < 3:
            continue
        
        file_url = parts[2]
        # We only want '.export.CSV.zip' (Events database). Skip .gkg and .mentions files.
        if ".export.CSV.zip" in file_url:
            filename = file_url.split("/")[-1]
            # GDELT 2.0 files start with YYYYMMDDHHMMSS
            file_timestamp = filename.split(".")[0]
            if file_timestamp.startswith(TARGET_YEAR_MONTH):
                target_urls.append(file_url)
    return target_urls

def process_filter(url):
    """Downloads a 15-min zip file, extracts it in memory, and filters for Algeria."""
    try:
        response = requests.get(url, timeout=30)
        if response.status_code != 200:
            return None
        
        # Unzip directly from bytes stream
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            file_inside = z.namelist()[0]
            with z.open(file_inside) as f:
                # Read CSV; GDELT is strictly tab-delimited (\t) without headers
                df = pd.read_csv(f, sep='\t', names=GDELT_HEADERS, header=None, dtype=str)
                
                # Apply geographical/actor filtering rules for Morocco
                morocco_mask = (
                    (df["Actor1CountryCode"] == MOROCCO_CAMEO_CODE) |
                    (df["Actor2CountryCode"] == MOROCCO_CAMEO_CODE) |
                    (df["ActionGeo_CountryCode"] == MOROCCO_FIPS_CODE)
                )
                return df[morocco_mask]
    except Exception as e:
        print(f"Error processing {url}: {e}")
        return None

def main():
    lines = get_file_list()
    target_urls = filter_events(lines)
    
    total_files = len(target_urls)
    print(f"Found {total_files} export files matching for this period: {TARGET_YEAR_MONTH}.")
    
    first_chunk = True
    processed_count = 0
    
    # Iterate through each 15-minute file chunk
    for url in target_urls:
        processed_count += 1
        if processed_count % 50 == 0 or processed_count == total_files:
            print(f"Processing progress: {processed_count}/{total_files} files...")
            
        filtered_df = process_filter(url)
        
        if filtered_df is not None and not filtered_df.empty:
            # Append directly to local storage to be memory-efficient
            if first_chunk:
                filtered_df.to_csv(OUTPUT_MASTER_CSV, index=False, mode='w', header=True)
                first_chunk = False
            else:
                filtered_df.to_csv(OUTPUT_MASTER_CSV, index=False, mode='a', header=False)
                
    print(f"Pipeline complete! Execution finished. File saved as: {OUTPUT_MASTER_CSV}")

if __name__ == "__main__":
    main()

# **2. CLEANING**

### **1. General Cleaning**

In [ ]:
# IMPORT CLEANING TABLES : 
domain = pd.read_csv(r"C:\Users\barba\OneDrive\Documents\Learning\GDELT\domains.csv", index_col=False)
event_code = pd.read_csv(r"C:\Users\barba\OneDrive\Documents\Learning\GDELT\Event_Code.csv", index_col=False)

In [5]:
# CREATING CLEANING DICTIONARIES
quadname = { 1: "Verbal Cooperation", 2: "Material Cooperation", 3: "Verbal Conflict", 4: "Material Conflict"}
country_codes = {"AFG": "Afghanistan", "AFR": "Africa", "ALA": "Aland Islands", "ALB": "Albania", "DZA": "Algeria", "ASM": "American Samoa",    "AND": "Andorra",    "AGO": "Angola",    "AIA": "Anguilla",
                "ATG": "Antigua and Barbuda",    "ARG": "Argentina",    "ARM": "Armenia",    "ABW": "Aruba",    "ASA": "Asia",    "AUS": "Australia",    "AUT": "Austria",    "AZE": "Azerbaijan",    "BAG": "Baghdad",    "BHS": "Bahamas",    "BHR": "Bahrain",    "BLK": "Balkans",    "BGD": "Bangladesh",    "BRB": "Barbados",    "BLR": "Belarus",    "BEL": "Belgium",    "BLZ": "Belize",    "BEN": "Benin",    "BMU": "Bermuda",    "BTN": "Bhutan",    "BOL": "Bolivia",    "BIH": "Bosnia and Herzegovina",    "BWA": "Botswana",    "BRA": "Brazil",    "VGB": "British Virgin Islands",    "BRN": "Brunei Darussalam",    "BGR": "Bulgaria",    "BFA": "Burkina Faso",    "BDI": "Burundi",    "KHM": "Cambodia",    "CMR": "Cameroon",    "CAN": "Canada",    "CPV": "Cape Verde",    "CRB": "Caribbean",    "CAU": "Caucasus",    "CYM": "Cayman Islands",    "CFR": "Central Africa",    "CAF": "Central African Republic",    "CAS": "Central Asia",    "CEU": "Central Europe",    "TCD": "Chad",    "CHL": "Chile",    "CHN": "China",    "COL": "Columbia",    "COM": "Comoros",    "COK": "Cook Islands",    "CRI": "Costa Rica",    "HRV": "Croatia",    "CUB": "Cuba",    "CYP": "Cyprus",    "CZE": "Czech Republic",    "COD": "Democratic Republic of the Congo",    "DNK": "Denmark",    "DJI": "Djibouti",    "DMA": "Dominica",    "DOM": "Dominican Republic",    "EIN": "East Indies",    "TMP": "East Timor",    "EAF": "Eastern Africa",    "EEU": "Eastern Europe",    "ECU": "Ecuador",    "EGY": "Egypt",    "SLV": "El Salvador",    "GNQ": "Equatorial Guinea",    "ERI": "Eritrea",    "EST": "Estonia",    "ETH": "Ethiopia",    "EUR": "Europe",    "FRO": "Faeroe Islands",
                "FLK": "Falkland Islands",    "FJI": "Fiji",    "FIN": "Finland",    "FRA": "France",    "GUF": "French Guiana",    "PYF": "French Polynesia",    "GAB": "Gabon",    "GMB": "Gambia",    "GZS": "Gaza Strip",    "GEO": "Georgia",    "DEU": "Germany",    "GHA": "Ghana",    "GIB": "Gibraltar",    "GRC": "Greece",    "GRL": "Greenland",    "GRD": "Grenada",    "GLP": "Guadeloupe",    "GUM": "Guam",    "GTM": "Guatemala",    "GIN": "Guinea",    "GNB": "Guinea-Bissau",    "GUY": "Guyana",    "HTI": "Haiti",    "HND": "Honduras",    "HKG": "Hong Kong",    "HUN": "Hungary",    "ISL": "Iceland",    "IND": "India",    "IDN": "Indonesia",    "IRN": "Iran",    "IRQ": "Iraq",    "IRL": "Ireland",    "IMY": "Isle of Man",    "ISR": "Israel",    "ITA": "Italy",    "CIV": "Ivory Coast",
                "JAM": "Jamaica",    "JPN": "Japan",    "JOR": "Jordan",    "KAZ": "Kazakhstan",    "KEN": "Kenya",    "KIR": "Kiribati",    "KWT": "Kuwait",    "KGZ": "Kyrgyzstan",    "LAO": "Laos",    "LAM": "Latin America",    "LVA": "Latvia",    "LBN": "Lebanon",    "LSO": "Lesotho",    "LBR": "Liberia",    "LBY": "Libya",    "LIE": "Liechtenstein",    "LTU": "Lithuania",    "LUX": "Luxembourg",    "MAC": "Macao",    "MKD": "Macedonia",    "MDG": "Madagascar",    "MWI": "Malawi",    "MYS": "Malaysia",    "MDV": "Maldives",    "MLI": "Mali",    "MLT": "Malta",    "MHL": "Marshall Islands",    "MTQ": "Martinique",    "MRT": "Mauritania",    "MUS": "Mauritius",    "MYT": "Mayotte",    "MDT": "Mediterranean",    "MEX": "Mexico",    "FSM": "Micronesia",    "MEA": "Middle East",    "MDA": "Moldova",    "MCO": "Monaco",    "MNG": "Mongolia",    "MTN": "Montenegro",    "MSR": "Montserrat",    "MAR": "Morocco",    "MOZ": "Mozambique",    "MMR": "Myanmar",    "NAM": "Namibia",    "NRU": "Nauru",    "NPL": "Nepal",    "NLD": "Netherlands",    "ANT": "Netherlands Antilles",    "NCL": "New Caledonia",    "NZL": "New Zealand",    "NIC": "Nicaragua",    "NER": "Niger",    "NGA": "Nigeria",    "NIU": "Niue",    "NFK": "Norfolk Island",
                "NAF": "North Africa",    "NMR": "North America",    "PRK": "North Korea",    "MNP": "Northern Mariana Islands",    "NOR": "Norway",    "PSE": "Occupied Palestinian Territory",    "OMN": "Oman",    "PAK": "Pakistan",    "PLW": "Palau",    "PAN": "Panama",    "PNG": "Papua New Guinea",    "PRY": "Paraguay",    "COG": "People's Republic of the Congo",    "PGS": "Persian Gulf",    "PER": "Peru",    "PHL": "Philippines",    "PCN": "Pitcairn",    "POL": "Poland",    "PRT": "Portugal",    "PRI": "Puerto Rico",    "QAT": "Qatar",    "ROM": "Romania",    "REU": "Runion",    "RUS": "Russia",    "RWA": "Rwanda",    "SHN": "Saint Helena",    "KNA": "Saint Kitts-Nevis",    "LCA": "Saint Lucia",    "SPM": "Saint Pierre and Miquelon",    "VCT": "Saint Vincent and the Grenadines",
                "WSM": "Samoa",    "SMR": "San Marino",    "STP": "Sao Tome and Principe",    "SAU": "Saudi Arabia",    "SCN": "Scandinavia",    "SEN": "Senegal",    "SRB": "Serbia",    "SYC": "Seychelles",    "SLE": "Sierra Leone",    "SGP": "Singapore",    "SVK": "Slovakia",    "SVN": "Slovenia",    "SLB": "Solomon Islands",    "SOM": "Somalia",    "ZAF": "South Africa",    "SAM": "South America",    "SAS": "South Asia",    "KOR": "South Korea",    "SEA": "Southeast Asia",    "SAF": "Southern Africa",    "ESP": "Spain",    "LKA": "Sri Lanka",    "SDN": "Sudan",    "SUR": "Suriname",    "SJM": "Svalbard and Jan Mayen Islands",
                "SWZ": "Swaziland",    "SWE": "Sweden",    "CHE": "Switzerland",    "SYR": "Syria",    "TWN": "Taiwan",    "TJK": "Tajikistan",    "TZA": "Tanzania",    "THA": "Thailand",    "WST": "The West",    "TGO": "Togo",    "TKL": "Tokelau",    "TON": "Tonga",    "TTO": "Trinidad and Tobago",    "TUN": "Tunisia",    "TUR": "Turkey",    "TKM": "Turkmenistan",    "TCA": "Turks and Caicos Islands",    "TUV": "Tuvalu",    "UGA": "Uganda",    "UKR": "Ukraine",    "ARE": "United Arab Emirates",    "GBR": "United Kingdom",    "USA": "United States",    "VIR": "United States Virgin Islands",    "URY": "Uruguay",
                "UZB": "Uzbekistan",    "VUT": "Vanuatu",    "VAT": "Vatican City",    "VEN": "Venezuela",    "VNM": "Vietnam",    "WLF": "Wallis and Futuna Islands",    "WAF": "West Africa",
                "WSB": "West Bank",    "ESH": "Western Sahara",    "YEM": "Yemen",    "ZMB": "Zambia",    "ZWE": "Zimbabwe"}
religion_codes = religions = {"ADR": "African Diasporic Religion", "ALE": "Alewi", "ATH": "Agnostic", "BAH": "Bahai Faith", "BUD": "Buddhism","CHR": "Christianity","CON": "Confucianism", "CPT": "Coptic", "CTH": "Catholic",
                              "DOX": "Orthodox", "DRZ": "Druze", "HIN": "Hinduism", "HSD": "Hasidic", "ITR": "Indigenous Tribal Religion","JAN": "Jainism", "JEW": "Judaism", "JHW": "Jehovah's Witness", 
                              "LDS": "Latter Day Saints", "MOS": "Muslim", "MRN": "Maronite","NRM": "New Religious Movement", "PAG": "Pagan",  "PRO": "Protestant",  "SFI": "Sufi", "SHI": "Shia",
                              "SHN": "Old Shinto School", "SIK": "Sikh", "SUN": "Sunni", "TAO": "Taoist", "UDX": "Ultra-Orthodox", "ZRO": "Zoroastrianism"}
ethnic_codes = { "AFA": "Black African", "ARA": "Arab", "ARB": "Arab", "ATS": "Agnostic/Athiest", "BER": "Berber", "KAB": "Kabyle", "AAR": "Afar", "ABK": "Abkhaz", "ABR": "Aboriginal Australians", "ACE": "Acehnese", "ACG": "Achang", "ACH": "Acholi",
                "ADA": "Ga", "ADI": "Adivasi", "ADJ": "Adjarians", "ADY": "Adyghe", "AFR": "Afrikaners", "AHM": "Ahmadis", "AIN": "Ainu", "AJA": "Aja", "AKA": "Akan", "AKU": "Aku", "ALA": "Alawi", "ALB": "Albanian","ALE": "Aleut", "ALG": "Algonquian", "ALT": "Altay", "ALU": "Alur", "AMB": "Ambonese", "AME": "Americo Liberians", "AMH": "Amhara", "ANP": "Angika speakers", "APA": "Apache", "ARG": "Aragonese", "ARM": "Armenian", "ARN": "Mapuche",
                "ARP": "Arapaho", "ARW": "Arawak", "ASA": "Asian", "ASH": "Ashkenazi Jews", "ASM": "Assamese", "AST": "Asturian", "ASY": "Assyrian", "ATA": "Atacamenos", "ATG": "Argentinians", "ATH": "Athabaskan", "AUS": "Australians", "AUU": "Austrians", "AVA": "Caucasian Avars", "AWA": "Awadhi", "AYM": "Aymara", "AZE": "Azerbaijani", "BAD": "Baganda", "BAH": "Bahais", "BAI": "Bamileke", "BAK": "Bashkirs", "BAL": "Baloch", "BAM": "Bambara", "BAN": "Balinese", "BAQ": "Basque",
                "BAR": "Bari", "BAS": "Basoga", "BAY": "Gbaya", "BDA": "Rakhine", "BEJ": "Beja", "BEL": "Belarusians", "BEM": "Bemba", "BEN": "Bengali Hindu", "BEY": "Beydan", "BHO": "Bhojpuri", "BIH": "Bihari", "BII": "Bai",  "BIK": "Bicolano", "BIN": "Edo", "BIS": "Urban ni Vanautu", "BKE": "Bateke", "BKN": "Bakongo", "BKW": "Bakweri", "BLA": "Siksikawa", "BLG": "Blang", "BLK": "Balkars", "BLN": "Balanta", "BMR": "Bamar", "BNI": "Beni Shugal Gumez",
                "BNT": "Bantu", "BNY": "Banyarwanda", "BOD": "Tibetan", "BOL": "Bolivia", "BON": "Bonan", "BOS": "Bosniaks", "BOU": "Buyei", "BRA": "Brijwasi", "BRB": "Bariba", "BRE": "Breton", "BRH": "Brahui", "BRK": "Burakumin",  "BRM": "Kurichiya", "BSH": "Bushmen", "BST": "Baster", "BSU": "Subiya", "BTE": "Beti Pahuin", "BTK": "Batak", "BUA": "Buryat", "BUD": "Buddhist", "BUG": "Bugis", "BUL": "Bulgarian", "BYN": "Bilen", "CAB": "Cabindan Mayombe",
                "CAD": "Caddo", "CAP": "Cape Verdean", "CAR": "Kali'na", "CAT": "Catalan", "CEB": "Cebuano", "CHA": "Chamorro", "CHC": "Chukchi", "CHE": "Chechen", "CHG": "Chagatai", "CHI": "Chinese", "CHK": "Chuukese", "CHL": "Chileans",  "CHM": "Mari", "CHN": "Chinook", "CHO": "Choctaw", "CHP": "Chipewyan", "CHR": "Cherokee", "CHT": "Ch'orti'", "CHV": "Chuvash", "CHW": "Chewa", "CHY": "Cheyenne", "CIR": "Adyghe", "CMC": "Cham", "COL": "Colombian",
                "CON": "Confusian", "COP": "Coptic Christians", "COR": "Cornish", "COS": "Corsican", "COT": "Cotiers", "CPE": "English Creole", "CPF": "French Creole", "CPP": "Portuguese Creole", "CRE": "Cree", "CRH": "Crimean Tatar", "CRI": "Christian", "CRO": "Orthodox Christian",
                "CRP": "Creole", "CSB": "Kashubian", "CSR": "Costa Ricans", "CTH": "Catholics", "CUS": "Cushitic", "CZE": "Czech", "DAI": "Dai", "DAK": "Sioux", "DAL": "Dalit", "DAM": "Damara", "DAN": "Danes", "DAO": "Yao (Asia)",  "DAR": "Dargwa", "DAU": "Daur", "DAY": "Dayak", "DEL": "Lenape", "DEN": "Slavey", "DGR": "Dogrib", "DIN": "Dinka", "DIV": "Maldivian", "DJE": "Djerma Songhai", "DOI": "Dogras", "DOM": "Dominicans", "DON": "Dong",
                "DOX": "Dongxiang", "DRA": "Dravidian", "DRU": "Druze", "DRZ": "Druze", "DSB": "Lower Sorbian", "DUA": "Duala", "DUT": "Dutch", "DYU": "Dyula", "DZO": "Ngalop", "EAT": "East Timorese", "ECU": "Ecuadorians", "EFI": "Efik",  "EIN": "East Indian", "EKA": "Ekajuk", "ENG": "English", "ESH": "Eshira", "EST": "Estonian", "ETH": "Ethiopian Jews", "EUR": "Europeans", "EVE": "Evenks", "EWE": "Ewe", "EWO": "Ewondo", "FAN": "Fang", "FAO": "Faroese",
                "FAT": "Fante", "FIJ": "Fijian", "FIL": "Filipino", "FIN": "Finns", "FIU": "Finno Ugric", "FON": "Fon", "FRE": "French", "FRI": "Santals", "FRR": "Frisians", "FRU": "Fur", "FUL": "Fula", "FUR": "Friulan",    "GAR": "Garifuna", "GAY": "Gayo", "GBA": "Gbaya", "GEL": "Gelao", "GEO": "Georgian", "GER": "German", "GIA": "Gia Rai", "GIL": "Kiribati", "GIN": "Gin", "GIO": "Gio", "GLA": "Gaels", "GLE": "Irish",
                "GLG": "Galician", "GLV": "Manx", "GON": "Gondi", "GOR": "Gorontalonese", "GRA": "Grassfielders", "GRB": "Grebo", "GRE": "Greek", "GRN": "Guarani", "GSW": "Swiss Germans", "GUA": "Guatemalan", "GUJ": "Gujarati", "GUN": "Guan",  "GWI": "Gwich'in", "HAD": "Hadjerai", "HAI": "Haida", "HAR": "Harari", "HAT": "Haitian", "HAU": "Hausa", "HAW": "Hawaiian", "HAZ": "Hazara", "HER": "Herero", "HGH": "Hill Tribes", "HIL": "Hiligayon", "HIM": "Himachali",
                "HIN": "Hindu", "HJW": "Hasidic", "HMN": "Hmong", "HMO": "Hiri Motu", "HNI": "Hani", "HOA": "Hoa", "HON": "Hondurans", "HRT": "Haratin", "HRV": "Croats", "HSB": "Upper Sorbian", "HUI": "Hui", "HUN": "Hungarian", "HUP": "Hupa", "HUT": "Hutu", "IBA": "Iban", "IBO": "Igbo", "ICE": "Icelanders", "IDG": "Indigenous", "IDN": "Indian", "III": "Yi", "IJO": "Ijaw", "IKU": "Inuit", "ILO": "Ilocono", "IND": "Indonesian",
                "INH": "Ingush", "IPK": "Inupiat", "IRA": "Iranian", "IRO": "Iroquois", "ITA": "Itallian", "JAN": "Jain", "JAV": "Javanese", "JEW": "Jewish", "JHW": "Jehovah's Witnesses", "JIN": "Jino", "JOL": "Jola", "JPN": "Japanese", "KAA": "Karakalpak", "KAC": "Kachin", "KAD": "Kadazan", "KAK": "Kakwa Nubian", "KAL": "Kalaallit", "KAM": "Kamba", "KAN": "Kannada", "KAO": "Kaonde", "KAR": "Karen", "KAS": "Kashmiri", "KAU": "Kanuri", "KAV": "Kavango",
                "KAZ": "Kazakhs", "KBD": "Kabarday", "KBY": "Kabye", "KCH": "Karachays", "KHA": "Khasi", "KHI": "Khoikhoi", "KHK": "Khakas", "KHM": "Khmer", "KHU": "Khmu", "KIK": "Kikuyu", "KIN": "Kinyarwanda Speakers", "KIR": "Kyrgyz", "KIS": "Kisii", "KLM": "Kalmyk", "KMB": "North Mbundu", "KNO": "Kono", "KNR": "Kanuri", "KOK": "Kokani", "KOM": "Komi", "KON": "Kongo", "KOR": "Korean", "KOS": "Kosraean", "KOU": "Kouyou", "KPE": "Kpelle",
                "KRH": "Krahn", "KRL": "Karelians", "KRM": "Karamojong", "KRO": "Kru", "KRU": "Kurukh", "KUA": "Kwanyama", "KUM": "Kumyks", "KUR": "Kurd", "KUT": "Ktunaxa", "LAD": "Sephardic Jew", "LAK": "Lak (Russia)", "LAM": "Lamba", "LAO": "Lao", "LAR": "Lari", "LAV": "Latvian", "LBA": "Limba", "LDS": "Latter Day Saints", "LEN": "Lenca", "LEZ": "Lezgian", "LGB": "Lugbara", "LHU": "Lahu", "LII": "Li", "LIM": "Limburgian", "LIN": "Lingala",
                "LIT": "Lithuanian", "LOL": "Mongo", "LOM": "Lomwe", "LOV": "Lovale", "LOZ": "Lozi", "LSU": "Lisu", "LTK": "Latoka", "LTN": "Latinos", "LTZ": "Luxembourgers", "LUA": "Luba Kasai", "LUB": "Luba Katanga", "LUG": "Baganda", "LUH": "Luhya", "LUI": "Luiseno", "LUL": "Lulua", "LUN": "Lunda", "LUO": "Luo", "LUS": "Lusei", "MAC": "Macedonian", "MAD": "Madurese", "MAF": "Mafwe", "MAG": "Magahi", "MAH": "Marshallese", "MAI": "Maithili",
                "MAK": "Makassarese", "MAL": "Malayalam", "MAN": "Mandinka", "MAO": "Maori", "MAR": "Marathi", "MAS": "Maasai", "MAY": "Malays", "MBA": "Mbandja", "MBE": "Mbere", "MBK": "M'Baka", "MBO": "Mbochi", "MBU": "Mbundu Mestico", "MDF": "Mokshas", "MDH": "Madhesi", "MDI": "Madi", "MDR": "Mandar", "MEN": "Mende", "MIA": "Miao", "MIC": "Mi'kmaq", "MIJ": "Mijikenda", "MIN": "Minangkabau", "MIZ": "Mizo", "MLA": "Mulatto", "MLD": "Mole Dagbani",
                "MLG": "Malagasy", "MLO": "Mulao", "MLT": "Maltese", "MNC": "Manchu", "MND": "Mande", "MNG": "Mananja Nayanja", "MNH": "Minahasa", "MNI": "Manipuri", "MNJ": "Manjack", "MNN": "Mano", "MNO": "Lumad", "MNS": "Mon", "MNY": "Manyika", "MOH": "Mohawk", "MOK": "Makonde", "MON": "Mongol", "MOS": "Mossi", "MRI": "Mari", "MRN": "Maronites", "MRO": "Moro", "MSK": "Miskito", "MSL": "Muslim", "MTN": "Montenegrins", "MTZ": "Mestizo",
                "MUN": "Munda", "MUO": "Muong", "MUS": "Muscogee", "MWL": "Mirandese", "MWR": "Marwaris", "MYA": "Mayangnas", "MYE": "Myene", "MYN": "Maya", "MYV": "Mordvins", "NAG": "Naga", "NAH": "Nahua", "NAI": "Native American", "NAM": "Nama", "NAP": "Neapolitan", "NAU": "Nauruan", "NAV": "Navajo", "NAX": "Nakhi", "NBA": "Nuba", "NBL": "South Ndebele", "NCA": "Nicaraguan", "NDE": "Northern Ndebele", "NDO": "Ndonga", "NEP": "Nepali", "NER": "Nuer",
                "NEW": "Newars", "NGN": "Ngbandi", "NGO": "Ngoni", "NIA": "Niasans", "NIB": "Nibolek", "NIR": "Niari", "NIU": "Niuean", "NKM": "Nkomi", "NNG": "Nung", "NOG": "Nogais", "NOR": "Norwegians", "NSO": "Northern Sotho", "NUB": "Nubian", "NUR": "Nuristani", "NUU": "Nu", "NYA": "Chewa", "NYK": "Nyakyusa", "NYM": "Nyamwezi", "NYN": "Ankole", "NYO": "Nyoro", "NZE": "New Zealanders", "NZI": "Nzema", "OCI": "Occitanians", "OGO": "Ogoni",
                "OJI": "Ojibwe", "OJW": "Orthodox/Ultra Orthodox Jew", "OKI": "Okinawan", "ORI": "Oriya", "ORM": "Oromo", "ORU": "Orgunu", "OSA": "Osage", "OSS": "Ossetians", "OTO": "Otomi", "OVA": "Ovambo", "PAA": "Papuan", "PAC": "Pacific Islanders", "PAG": "Pangasinan", "PAL": "Palestinian", "PAM": "Kapampangan", "PAN": "Punjabi", "PAP": "Papiamento Creole", "PAR": "Paraguayan", "PAU": "Palauan", "PER": "Persian", "PGN": "Animist/Pagan", "PHU": "Puthai", "PNM": "Panamanians", "POL": "Poles",
                "POM": "Pomaks", "PON": "Pehnpeian", "POR": "Portuguese", "PPL": "Papel", "PRO": "Protestant", "PRU": "Peruvian", "PSH": "Pashayi", "PUM": "Pumi", "PUS": "Pashtun", "QIA": "Qiang", "QIZ": "Qizilbash", "QUE": "Quechua","RAJ": "Rajasthani", "RAN": "Pahari Rajput", "RAP": "Rapa Nui", "RAR": "Cook Islands Maori", "REL": "Unspecified Religion", "ROH": "Romansh", "ROM": "Romani", "RUM": "Romanian", "RUN": "Rundi", "RUP": "Aromanians", "RUS": "Russian", "SAD": "Sandawe",
                "SAG": "Sango", "SAH": "Yakuts", "SAL": "Salish", "SAR": "Sara", "SAS": "Sasak", "SAT": "Sudanese", "SCN": "Sicilian", "SCO": "Scottish", "SEL": "Selkup", "SEN": "Sena", "SFI": "Sufi", "SHA": "Shafi'i", "SHE": "She", "SHI": "Shi'ites", "SHL": "Shilluk", "SHN": "Shan", "SHY": "Shaigiya", "SID": "Sidama", "SIN": "Sinhalese", "SIO": "Siouan", "SLA": "Slavic", "SLO": "Slovaks", "SLR": "Salar", "SLV": "Slovenes",
                "SMI": "Sami", "SMO": "Samoans", "SNA": "Shona", "SND": "Sindhi", "SNK": "Soninke", "SOM": "Somali", "SON": "Songhai", "SOT": "Sotho", "SPA": "Spanish", "SRD": "Sardinian", "SRN": "Sranan Tongo", "SRP": "Serbs", "SRR": "Serer", "SSW": "Swazi", "SUI": "Sui", "SUK": "Sukama", "SUN": "Sunni", "SUS": "Susu", "SWA": "Swahili", "SWE": "Swedes", "SWF": "Swiss French", "SWT": "Swiss Italian", "TAB": "Tabasaran", "TAH": "Tahitian",
                "TAI": "Tai", "TAM": "Tamil", "TAO": "Taoist", "TAT": "Tatars", "TAW": "Tawahka", "TAY": "Tay", "TEL": "Telugu", "TEM": "Temne", "TER": "Terenan", "TES": "Teso", "TET": "Tetum", "TGK": "Tajik", "TGL": "Tagalog", "THA": "Thai", "TIB": "Tibetan", "TIG": "Tigre", "TIR": "Tigray Tigrinya", "TIV": "Tiv", "TKL": "Tokelauan", "TLI": "Tlingit", "TMH": "Tuareg", "TMS": "Tama", "TOG": "Tonga (Africa)", "TON": "Tonga (Pacific)",
                "TOR": "Tooro", "TOU": "Toubou", "TPI": "Tok Pisin", "TRA": "Transnistrians", "TRI": "Tripuri", "TRN": "Ternate", "TSI": "Tsimshian", "TSN": "Tswana", "TSO": "Tsonga", "TTS": "Tutsi", "TUJ": "Tujia", "TUK": "Turkmen", "TUM": "Tumbuka", "TUP": "Tupi", "TUR": "Turkish", "TUU": "Mongour", "TVL": "Tuvaluans", "TWI": "Ashanti", "TWN": "Taiwanese", "TYV": "Tuvans", "UDM": "Udmurt", "UIG": "Uyghur", "UKR": "Ukranian", "UMB": "Southern Mbundu",
                "UND": "Undetermined", "URD": "Urdu", "UZB": "Uzbeks", "VAA": "Va", "VAI": "Vai", "VEN": "Venda", "VIE": "Vietnamese", "VIL": "Vili", "VNZ": "Venezuelan", "VOT": "Votes", "WAK": "Wakashan", "WAL": "Welayta","WAR": "Waray", "WAS": "Washoe", "WEL": "Welsh", "WEN": "Sorbs", "WHI": "Whites", "WLN": "Walloons", "WOL": "Wolof", "XAL": "Kalmyk", "XHO": "Xhosa", "XIB": "Xibe", "XNC": "Xinca", "YAO": "Yao",
                "YAP": "Yapese", "YOR": "Yoruba", "YPK": "Yupik", "YUG": "Yugur", "ZAG": "Zaghawa", "ZAP": "Zapotec", "ZAY": "Zaidiyya", "ZEN": "Zenaga", "ZHA": "Zhuang", "ZND": "Azande", "ZOM": "Zomi", "ZOR": "Zoroastrians", "ZUL": "Zulu", "ZUN": "Zuni", "ZZA": "Zaza"}



In [ ]:
#1 IMPORT THE DATAFRAME
df = pd.read_csv(OUTPUT_MASTER_CSV)
print(len(df))
df["Day"] = df["SQLDATE"].apply(lambda x: str(x)[6:8])  # Extract day from SQLDATE
df["Month"] = df["SQLDATE"].apply(lambda x: str(x)[4:6])  # Extract month from SQLDATE
#2 DATE VERIFICATION  
df = df[(df["Month"]==str(TARGET_YEAR_MONTH[4:6])) & (df["Year"]==int(TARGET_YEAR_MONTH[0:4]))] 
#3 DROP EXTRA DATE COLUMNS :SQLDATE, MonthYear, FractionDate, DATEADDED
df = df.drop(columns=["SQLDATE", "MonthYear", "FractionDate", "DATEADDED"])
#3 DROP DUPLICATE ROWS
df = df.drop_duplicates()
print(len(df))
df.to_csv(f"MAR{TARGET_YEAR_MONTH}Cleaned.csv", index=False)

12923
12866


### **2. Media Outlet Analysis**

In [7]:
#1 KEEP ONLY COLUMNS OF INTEREST
media = df[['Day', 'Month', 'Year', 'SOURCEURL', 'AvgTone']]
#2.1 DROP DUPLICATE ROWS IN SOURCEURL COLUMN
media = media.drop_duplicates()
#3 GROUP BY SOURCEURL AND AVG(AVGTONE)
media = media.groupby('SOURCEURL').agg({'Day': 'first', 
                                        'Month': 'first',
                                        'Year': 'first',
                                        'AvgTone': 'mean'}).reset_index()
#2.2 REMOVE DUPLICATES AGAIN
media = media.drop_duplicates()
#4 ADD COLUM DOMAIN 
media['Domain'] = media['SOURCEURL'].apply(lambda x: re.match(r"https?://(?:www\.)?([^/]+)", x).group(1))
#5 ADD CLEANING LAYER : REMOVE :443 AT THE END OF THE DOMAIN
media['Domain'] = media['Domain'].apply(lambda x: x.replace(':443', ''))
#6 EXPORT 
media.to_csv(f"media{TARGET_YEAR_MONTH}.csv", index=False)

### **3. Subnational Analysis**

In [ ]:
#1 CREATE A FUNCTION TO EXTRACT THE FIRST STRING BEFORE THE FIRST COMMA IN ACTIONGEO_FULLNAME
def extract_first_string(fullname):
    if pd.isnull(fullname):
        return None
    return fullname.split(',')[0].strip()
#2 CREATE A DICTIONARY OF REGIONS : 
ma_regions = {"MO57": "Tanger-Tétouan", "MO54": "L'Oriental", "MO49": "Rabat-Salé-Zemmour-Zaër", "MO56": "Tadla-Azilal", 
           "MO51": "Doukkala-Abda", "MO45": "Grand Casablanca", "MO52": "Gharb-Chrarda-Beni Hssen", "MO55": "Souss-Massa-Drâa", 
           "MO50": "Chaouia-Ouardigha", "MO47": "Marrakech-Tensift-Al Haouz", "MO48": "Meknès-Tafilalet", 
           "MO58": "Taza-Al Hoceïma-Taounate", "MO53": "Guelmim-Es Semara", "MO00": "Morocco", "MO46": "Fès-Boulemane"}
#3 CREATE A DATAFRAME WITH COLUMNS OF INTEREST 
locality = df[['Day', 'Month', 'Year','EventBaseCode',  'GoldsteinScale','AvgTone', 'ActionGeo_Type', 
               'ActionGeo_FullName', 'ActionGeo_CountryCode', 'ActionGeo_ADM1Code', 'ActionGeo_ADM2Code', 
               'ActionGeo_Lat', 'ActionGeo_Long', 'ActionGeo_FeatureID', 'SOURCEURL']]
#4 FILTERING THE DATAFRAME (KEEP ONLY MO AND LOCAL LEVEL 4)
locality = locality[(locality['ActionGeo_CountryCode'] == 'MO') & (locality['ActionGeo_Type'] == 4)]
#5 CREATE A NEW COLUMN 'City' BY APPLYING THE FUNCTION TO 'ActionGeo_FullName'
locality['City'] = locality['ActionGeo_FullName'].apply(extract_first_string)
#6 CREATE A NEW COLUMN 'Region' BY MAPPING 'ActionGeo_ADM1Code' TO THE DICTIONARY OF REGIONS
locality['Region'] = locality['ActionGeo_ADM1Code'].map(ma_regions)
#7 REMOVE DUPLICATES
locality = locality.drop_duplicates()
#8 EXPORT THE DATAFRAME 
locality.to_csv(f"locality{TARGET_YEAR_MONTH}.csv", index=False)

### **4. Event Analysis**

In [18]:

#1 CREATE THE DATAFRAME OF EVENTS WITH COLUMNS OF INTEREST
event = df[['Day', 'Month', 'Year', 'Actor1CountryCode', 'Actor2CountryCode', 'ActionGeo_CountryCode', 'EventBaseCode', 'QuadClass', 'GoldsteinScale',  'AvgTone', 'SOURCEURL']]
#2 FILTER ALGERIA AS ACTOR
event = event[(event['Actor1CountryCode'] == 'DZA') | (event['Actor2CountryCode'] == 'DZA') ]
#3 MAP QUADCLASS TO NAMES
event['QuadClass'] = event['QuadClass'].map(quadname)
#4 GET EVENT NAMES FROM event_code TABLE
event = event.merge(event_code[['CAMEOEVENTCODE', 'EVENTDESCRIPTION']], how='left', left_on='EventBaseCode', right_on='CAMEOEVENTCODE')
#5 MAP ACTOR1 AND ACTOR2 COUNTRY CODES TO NAMES
event['Actor1'] = event['Actor1CountryCode'].map(country_codes)
event['Actor2'] = event['Actor2CountryCode'].map(country_codes)
event = event.drop(columns=['Actor1CountryCode', 'Actor2CountryCode']).reset_index(drop=True)
#6 REMOVE DUPLICATES 
event = event.drop_duplicates()
#7 EXPORT THE DATAFRAME 
event.to_csv(f"event{TARGET_YEAR_MONTH}.csv", index=False)

### **5. Geopolitics Analysis**

In [16]:
#1 KEEP ONLY COLUMNS OF INTEREST FOR GEOPOLITICS ANALYSIS 
geopolitics = df[['Day', 'Month', 'Year', 'Actor1CountryCode', 'Actor2CountryCode', 'IsRootEvent', 'EventBaseCode', 'QuadClass', 'GoldsteinScale', 'AvgTone', 'SOURCEURL']]
#2 DROP DUPLICATES 
geopolitics = geopolitics.drop_duplicates()
#3 FILTER FOR ALGERIA OR DZA  
geopolitics = geopolitics[(geopolitics['Actor1CountryCode'] == 'DZA') | (geopolitics['Actor2CountryCode'] == 'DZA')]
#4 MAP ACTOR1 AND ACTOR2 COUNTRY CODES TO NAMES
geopolitics['Actor1'] = geopolitics['Actor1CountryCode'].map(country_codes)
geopolitics['Actor2'] = geopolitics['Actor2CountryCode'].map(country_codes)
#5 DROP COLUMNS Actor1CountryCode AND Actor2CountryCode AND RESET INDEX 
geopolitics = geopolitics.drop(columns=['Actor1CountryCode', 'Actor2CountryCode']).reset_index(drop=True)
#6 Export the geopolitics dataframe
geopolitics.to_csv(f"geopolitics{TARGET_YEAR_MONTH}.csv", index=False)

### **6. Minority Analysis (Religious)**

In [11]:
#1 KEEP ONLY COLUMNS OF INTEREST FOR MINORITY RELIGIOUS ANALYSIS
minority_religious = df[['Day', 'Month', 'Year', 'Actor1CountryCode', 'Actor1Religion1Code', 'Actor1Religion2Code',
                         'Actor2CountryCode','Actor2Religion1Code', 'Actor2Religion2Code',
                         'EventBaseCode', 'QuadClass', 'GoldsteinScale', 'AvgTone', 'ActionGeo_CountryCode', 'SOURCEURL']]
#2 FILTER FOR MINORITY COLUMNS ARE NOT NULL 
minority_religious = minority_religious[(minority_religious['Actor1Religion1Code'].notnull()) | (minority_religious['Actor2Religion1Code'].notnull())]
#3 Map RELIGION CODES TO NAMES
minority_religious['Actor1Religion1Code'] = minority_religious['Actor1Religion1Code'].map(religion_codes)
minority_religious['Actor1Religion2Code'] = minority_religious['Actor1Religion2Code'].map(religion_codes)
minority_religious['Actor2Religion1Code'] = minority_religious['Actor2Religion1Code'].map(religion_codes)
minority_religious['Actor2Religion2Code'] = minority_religious['Actor2Religion2Code'].map(religion_codes)
#4 DROP DUPLICATES
minority_religious = minority_religious.drop_duplicates()
#5 EXPORT THE DATAFRAME
minority_religious.to_csv(f"minorityR{TARGET_YEAR_MONTH}.csv", index=False)

### **7. Minority Analysis (Ethnic)**

In [12]:
#1 KEEP ONLY COLUMNS OF INTEREST  
minority_ethnic = df[['Day', 'Month', 'Year','Actor1CountryCode', 'Actor1EthnicCode', 'Actor2CountryCode','Actor2EthnicCode', 
                       'EventBaseCode', 'QuadClass', 'GoldsteinScale', 'AvgTone', 'ActionGeo_CountryCode','SOURCEURL']]
#2 FILTER FOR MINORITY IS NOT NULL
minority_ethnic = minority_ethnic[(minority_ethnic['Actor1EthnicCode'].notnull()) | (minority_ethnic['Actor2EthnicCode'].notnull())]
# REPLACE ETHNIC CODE WITH FULL NAME USING THE DICTIONARY
minority_ethnic['Actor1EthnicCode'] = minority_ethnic['Actor1EthnicCode'].str.upper().map(ethnic_codes)
minority_ethnic['Actor2EthnicCode'] = minority_ethnic['Actor2EthnicCode'].str.upper().map(ethnic_codes)
#4 DROP DUPLICATES 
minority_ethnic = minority_ethnic.drop_duplicates()
#4 EXPORT THE DATAFRAME 
minority_ethnic.to_csv(f"minorityE{TARGET_YEAR_MONTH}.csv", index=False)

### **8. Chronology [URLs]**

In [13]:
#1 KEEP ONLY COLUMNS OF INTEREST
chronology = df[['Day', 'Month', 'Year', 'SOURCEURL', 'AvgTone']]
#2.1 DROP DUPLICATE ROWS IN SOURCEURL COLUMN
chronology = chronology.drop_duplicates()
#3 GROUP BY SOURCEURL AND AVG(AVGTONE)
chronology = chronology.groupby('SOURCEURL').agg({'Day': 'first', 
                                                  'Month': 'first', 
                                                  'Year': 'first',
                                                  'AvgTone': 'mean'}).reset_index()
#2.2 REMOVE DUPLICATES AGAIN
chronology = chronology.drop_duplicates()
#4 ADD COLUM DOMAIN 
chronology['Domain'] = chronology['SOURCEURL'].apply(lambda x: re.match(r"https?://(?:www\.)?([^/]+)", x).group(1))
#5 ADD CLEANING LAYER : REMOVE :443 AT THE END OF THE DOMAIN
chronology['Domain'] = chronology['Domain'].apply(lambda x: x.replace(':443', ''))
#6 EXPORT 
chronology.to_csv(f"chronology{TARGET_YEAR_MONTH}.csv", index=False)

# **3. EXPORT GOOGLE SHEET**

In [15]:
# 1. Authenticate with your GCP Service Account credentials
try:
    gc = gspread.service_account(filename=r"C:\Users\barba\OneDrive\Documents\Learning\GDELT\credentials.json")
    
    # Define your destination workbook name
    # Ensure you have shared this Sheet with your Service Account Email as an "Editor"!
    workbook_name = f"MAR{TARGET_YEAR_MONTH}"
    sh = gc.open(workbook_name)
    print(f"Connected to Google Sheet: {workbook_name}")
except Exception as e:
    print(f"Connection failed. Check your JSON file path or sharing permissions. Error: {e}")

# 2. Map your notebook's processed DataFrames to their respective Sheet Tab Names
# (Using the preserved 'Chronology' variable from the Section 8 fix)
gdelt_sheets_mapping = {
    "Cleaned": df,
    "Media": media,
    "Locality": locality,
    "Events": event,
    "Geopolitics": geopolitics,
    "Minority_Religious": minority_religious,
    "Minority_Ethnic": minority_ethnic,
    "Chronology": chronology,
    "Domain" : domain
}

# 3. Stream dataframes directly to individual tabs
for tab_name, dataframe in gdelt_sheets_mapping.items():
    print(f"Uploading data to tab: {tab_name}...")
    
    # Fetch or generate the worksheet tab safely
    try:
        worksheet = sh.worksheet(tab_name)
        worksheet.clear()  # Purge old structures/data points
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=tab_name, rows=1, cols=1)
        
    # Optimized batch upload
    set_with_dataframe(
        worksheet, 
        dataframe, 
        row=1, 
        col=1, 
        include_index=False, 
        include_column_header=True, 
        resize=True  # Automatically adjusts rows/columns to fit your data footprint
    )

print("\nAll GDELT data frames successfully exported into your Google Sheets workbook!")

Connected to Google Sheet: MAR202607
Uploading data to tab: Cleaned...
Uploading data to tab: Media...
Uploading data to tab: Locality...
Uploading data to tab: Events...
Uploading data to tab: Geopolitics...
Uploading data to tab: Minority_Religious...
Uploading data to tab: Minority_Ethnic...
Uploading data to tab: Chronology...
Uploading data to tab: Domain...

All GDELT data frames successfully exported into your Google Sheets workbook!
